In [ ]:
import pyspark.sql.functions as f

In [ ]:
def ingerir_camada_bronze(catalog_name: str, schema: str, entidades: list) -> None:
    """
    Lê arquivos CSV de Volumes do Databricks e ingere em tabelas Delta na camada Bronze.

    Args:
        catalog_name (str): Nome do catálogo do Unity Catalog.
        schema (str): Nome do database/schema do Unity Catalog.
        entidades (list): Lista de strings contendo os nomes dos diretórios/tabelas a 
                          serem processados (ex: ["vendas", "devolucoes"]).

    Returns:
        None. As tabelas são gravadas diretamente no Unity Catalog.
    """
    for dir in entidades:
        caminho_volume = f"/Volumes/{catalog_name}/{schema}/volume_arquivos_raw/{dir}"
        caminho_schema = f"/Volumes/{catalog_name}/{schema}/volume_arquivos_raw/_schemas/{dir}/"    
        caminho_checkpoint = f"/Volumes/{catalog_name}/{schema}/volume_arquivos_raw/_checkpoints/{dir}/"

        # Leitura dos arquivos CSV no volume
        df_bronze_csv = (
            spark.readStream
            .format("cloudFiles")
            .option("cloudFiles.format", "csv")
            .option("pathGlobFilter", "*.csv")
            .option("header", "true")
            .option("cloudFiles.includeExistingFiles", "true")
            .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
            .option("cloudFiles.inferColumnTypes", "true") # Deixa o Autoloader inferir os tipos brutos. _rescued_data é adicionada automaticamente pelo Auto Loader
            .option("cloudFiles.schemaLocation", caminho_schema)
            .option("cloudFiles.maxFilesPerTrigger", "3")
            .load(caminho_volume)
            .withColumn("_file_name", f.col("_metadata.file_path"))
            .withColumn("_ingested_at", f.current_timestamp())
        )

        # Escrita nas tabelas da camada Bronze exclusiva para os arquivos CSVs
        query = (
            df_bronze_csv
            .writeStream
            .format("delta")
            .outputMode("append")
            .option("mergeSchema", "true")
            .option("checkpointLocation", caminho_checkpoint)
            .trigger(availableNow=True)
            .toTable(f"{catalog_name}.{schema}.{dir}_stream")
        )

In [ ]:
catalog_name = dbutils.widgets.get("catalog_name")
schema = dbutils.widgets.get("schema")
lista_dir = ["vendas", "devolucoes", "movimentacao_estoque"]
ingerir_camada_bronze(catalog_name, schema, lista_dir)